# Track 1, Option A — Notebook 00: Baseline Evaluation

Evaluate the **base model** `Qwen/Qwen3-0.6B-Base` (pretrain-only) on the 10-prompt
manual test set using **BLEU** (sacrebleu) and **BERTScore (F1)**.

These baseline numbers are the reference point for the SFT and DPO notebooks.

**Kaggle setup:** GPU (T4 x2 or P100), Internet **ON**.

**Output:** `baseline_results.csv` plus example generations for the report.

In [14]:
%%capture
# Qwen3 needs transformers >= 4.51. Install a consistent, recent stack.
!pip install -q -U "transformers>=4.51.0" "trl>=0.15.0" "peft>=0.14.0" \
    "datasets>=3.2.0" "accelerate>=1.3.0" "evaluate>=0.4.3" \
    "sacrebleu>=2.4.3" "bert-score>=0.3.13" "sentencepiece>=0.2.0"

In [15]:
import os, json, random, numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# ---------------- Config ----------------
BASE_MODEL     = "Qwen/Qwen3-0.6B-Base"
SYSTEM_PROMPT  = "You are a helpful assistant. Answer concisely and directly."
MAX_NEW_TOKENS = 256
SEED           = 42
OUT_DIR        = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| dtype:", DTYPE)

# ChatML template (Qwen-style). The pretrain base ships the <|im_start|> /
# <|im_end|> tokens but no chat template, so we set one explicitly and reuse the
# SAME template across base / SFT / DPO for a fair comparison.
CHATML_TEMPLATE = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

device: cuda | dtype: torch.bfloat16


In [16]:
# ---------------- Test set (with inline fallback) ----------------
# Tries to load test_set.json from common Kaggle locations; otherwise uses the
# embedded copy so the notebook runs standalone.
INLINE_TEST_SET = {
    "system_prompt": SYSTEM_PROMPT,
    "prompts": [
        {"id": 1, "type": "factual_short", "difficulty": "easy", "prompt": "Which planet is known as the Red Planet? Answer in one short sentence.", "gold": "Mars is known as the Red Planet."},
        {"id": 2, "type": "sentiment_classification", "difficulty": "easy", "prompt": "Classify the sentiment of this sentence as positive or negative: 'I absolutely love this movie.'", "gold": "The sentiment is positive."},
        {"id": 3, "type": "math_reasoning", "difficulty": "easy", "prompt": "A train travels 60 km in 1.5 hours. What is its average speed in km/h?", "gold": "The average speed is 40 km/h."},
        {"id": 4, "type": "list_constrained", "difficulty": "medium", "prompt": "List exactly three primary colors, separated by commas.", "gold": "Red, yellow, blue."},
        {"id": 5, "type": "rewriting_tone", "difficulty": "medium", "prompt": "Rewrite this sentence to be more formal: 'Hey, can you send me that file ASAP?'", "gold": "Could you please send me that file at your earliest convenience?"},
        {"id": 6, "type": "summarization_grounded", "difficulty": "medium", "prompt": "Summarize the following text in one sentence: 'The library will be closed on Monday for a public holiday. It will reopen on Tuesday at 9 a.m. with normal hours.'", "gold": "The library is closed Monday for a public holiday and reopens Tuesday at 9 a.m."},
        {"id": 7, "type": "extraction", "difficulty": "medium", "prompt": "Extract when the library reopens from this text: 'The library will be closed on Monday for a public holiday. It will reopen on Tuesday at 9 a.m. with normal hours.'", "gold": "Tuesday at 9 a.m."},
        {"id": 8, "type": "sentiment_3class", "difficulty": "medium", "prompt": "Classify the following review as positive, negative, or neutral: 'The food was okay, nothing special.'", "gold": "The sentiment is neutral."},
        {"id": 9, "type": "unit_conversion", "difficulty": "medium", "prompt": "Convert 2 kilometers to meters.", "gold": "2 kilometers is 2000 meters."},
        {"id": 10, "type": "coding_oneliner", "difficulty": "medium", "prompt": "Write a single line of Python that computes the sum of a list named nums.", "gold": "total = sum(nums)"},
        {"id": 11, "type": "grammar", "difficulty": "medium", "prompt": "Give the past tense of the verb 'run' in a single word.", "gold": "Ran."},
        {"id": 12, "type": "translation", "difficulty": "medium", "prompt": "Translate 'Good morning' into French.", "gold": "Bonjour."},
        {"id": 13, "type": "date_reasoning", "difficulty": "harder", "prompt": "If today is Monday, what day will it be in 3 days?", "gold": "It will be Thursday."},
        {"id": 14, "type": "multi_step_math", "difficulty": "harder", "prompt": "A pizza is cut into 8 slices. If 3 people each eat 2 slices, how many slices remain?", "gold": "Two slices remain."},
        {"id": 15, "type": "explanation_audience", "difficulty": "harder", "prompt": "Explain what a CPU does in exactly two sentences.", "gold": "A CPU carries out the instructions of a program. It performs the calculations and logic operations that make a computer work."},
    ],
}

def load_test_set():
    for p in ["test_set.json", "/kaggle/input/test-set/test_set.json",
              "/kaggle/working/test_set.json"]:
        if os.path.exists(p):
            with open(p) as f:
                print("Loaded test set from", p)
                return json.load(f)
    print("Using inline test set.")
    return INLINE_TEST_SET

TEST_SET = load_test_set()
PROMPTS  = TEST_SET["prompts"]
print(f"{len(PROMPTS)} evaluation prompts.")

Using inline test set.
15 evaluation prompts.


In [18]:
# ---------------- Shared evaluation utilities ----------------
# (Copy this cell into the SFT / DPO notebooks too, or keep them self-contained.)
import sacrebleu
from bert_score import score as _bertscore
from tqdm.auto import tqdm

def load_tokenizer(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.chat_template = CHATML_TEMPLATE
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def build_prompt(tokenizer, user_msg, system_prompt=SYSTEM_PROMPT):
    msgs = ([{"role": "system", "content": system_prompt}] if system_prompt else [])
    msgs.append({"role": "user", "content": user_msg})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def _eos_ids(tokenizer):
    ids = []
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if isinstance(im_end, int) and im_end >= 0:
        ids.append(im_end)
    if tokenizer.eos_token_id is not None:
        ids.append(tokenizer.eos_token_id)
    return ids or None

@torch.no_grad()
def generate_response(model, tokenizer, user_msg, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(tokenizer, user_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=_eos_ids(tokenizer),
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def compute_bleu(pred, ref):
    return sacrebleu.sentence_bleu(pred, [ref]).score  # 0..100

def compute_bertscore_f1(preds, refs):
    _, _, f1 = _bertscore(preds, refs, lang="en", rescale_with_baseline=True)
    return [float(x) * 100 for x in f1]  # 0..100-ish

def evaluate_model(model, tokenizer, prompts, tag="model"):
    rows, preds, refs = [], [], []
    for p in tqdm(prompts, desc=f'generate [{tag}]'):
        pred = generate_response(model, tokenizer, p["prompt"])
        preds.append(pred); refs.append(p["gold"])
        rows.append({"id": p["id"], "type": p["type"], "prompt": p["prompt"],
                     "gold": p["gold"], "prediction": pred})
    bleus = [compute_bleu(pr, rf) for pr, rf in zip(preds, refs)]
    f1s   = compute_bertscore_f1(preds, refs)
    for r, b, f in zip(rows, bleus, f1s):
        r["bleu"], r["bertscore_f1"] = round(b, 3), round(f, 3)
    df = pd.DataFrame(rows)
    print(f"[{tag}] mean BLEU = {df.bleu.mean():.2f} | mean BERTScore-F1 = {df.bertscore_f1.mean():.2f}")
    return df



print("done")

done


In [19]:
# ---------------- Load base model + run baseline evaluation ----------------
tokenizer = load_tokenizer(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=DTYPE, device_map="auto"
)
model.eval()

baseline_df = evaluate_model(model, tokenizer, PROMPTS, tag="base")
baseline_df.to_csv(os.path.join(OUT_DIR, "baseline_results.csv"), index=False)

summary = {
    "model": BASE_MODEL,
    "mean_bleu": round(float(baseline_df.bleu.mean()), 3),
    "mean_bertscore_f1": round(float(baseline_df.bertscore_f1.mean()), 3),
}
with open(os.path.join(OUT_DIR, "baseline_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(summary)
baseline_df[["id", "type", "bleu", "bertscore_f1", "prediction"]]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[base] mean BLEU = 6.90 | mean BERTScore-F1 = -0.16
{'model': 'Qwen/Qwen3-0.6B-Base', 'mean_bleu': 6.896, 'mean_bertscore_f1': -0.16}


,id,type,bleu,bertscore_f1,prediction
0,1,factual_short,1.527,0.576,What is the capital city of France? Answer in ...
1,2,sentiment_classification,12.301,18.940,The sentiment of the sentence 'I absolutely lo...
2,3,math_reasoning,3.935,-34.560,"To find the average speed, we use the formula:..."
3,4,list_constrained,3.377,15.979,"List exactly three secondary colors, separated..."
4,5,rewriting_tone,13.265,19.118,"Rewrite this sentence to be more formal: 'Hey,..."
5,6,summarization_grounded,40.189,84.272,The library will be closed on Monday for a pub...
6,7,extraction,14.363,30.579,Extract when the library reopens from this tex...
7,8,sentiment_3class,1.693,11.091,The review is **negative**. The reviewer descr...
8,9,unit_conversion,0.252,-31.171,Convert 1000 meters to kilometers.__(*\nlarını...
9,10,coding_oneliner,0.175,-48.255,Write a single line of Python that computes th...


In [20]:
# ---------------- Save a few example generations for the report ----------------
with open(os.path.join(OUT_DIR, "sample_outputs.md"), "w") as f:
    f.write("# Sample outputs\n\n## Base model (Qwen3-0.6B-Base)\n\n")
    for _, r in baseline_df.head(5).iterrows():
        f.write(f"**Prompt ({r['type']}):** {r['prompt']}\n\n")
        f.write(f"- Gold: {r['gold']}\n")
        f.write(f"- Base (BLEU {r['bleu']}, BERTScore {r['bertscore_f1']}): {r['prediction']}\n\n")
print("Saved baseline_results.csv, baseline_summary.json, sample_outputs.md to", OUT_DIR)
print("Note: a pretrain-only base typically rambles / ignores the instruction —")
print("that is the expected weak baseline that SFT should improve on.")

Saved baseline_results.csv, baseline_summary.json, sample_outputs.md to /kaggle/working
Note: a pretrain-only base typically rambles / ignores the instruction —
that is the expected weak baseline that SFT should improve on.
